In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

<font size="5" color="black">ch3. Chat completions API</font>

# OpenAI Chat Completions API 기본 

- OpenAI의 Chat Completions API를 활용
    - GPT-4o-mini / gpt-4.1-nano

### 주요 학습 내용
1. OpenAI API 소개 및 환경 설정 : OpenAI API 개요, API 키 발급 및 보안 설정, 파이썬 클라이언트 설치 및 인스턴스 생성 방법
2. 기본적인 Chat Completions API 사용법 : 간단한 대화형 텍스트 생성 요청과 응답 처리, 프롬프트 엔지니어링 기초
3. 스트리밍 응답 : 대화 응답을 스트리밍 방식으로 받아 실시간 처리하는 방법
4. 시스템 메시지 활용 : 시스템 역할 메시지를 사용하여 AI의 응답 스타일이나 행동을 조정하는 방법
5. 고급 활용법 : 토큰 최적화와 비용 절감 전략, OpenAI API 에러 처리 및 예외 Handling
6. 실전 프로젝트 예제 : 간단한 챗봇 구현 및 외부 데이터/API와 연동하여 데이터 분석 기능을 결합한 사례

# 1. OpenAI API 소개 및 환경 설정


## 1.1 OpenAI API 개요
- OpenAI API : GPT 계열의 대규모 언어 모델을 인터넷을 통해 사용할 수 있도록 제공하는 서비스
- Chat Completions API
    - 챗봇과 유사한 대화형 상호작용을 할 수 있는 엔드포인트
    - 역할(role)이 부여된 메시지 목록을 입력하면 모델이 다음 대화 내용을 생성 
    - GPT-4o는 텍스트와 이미지 입력을 모두 처리하며 최대 128k 토큰 가용

## 1.2 API 키 발급 및 보안 설정
- OpenAI 계정에서 API 키 발급
    * OpenAI - API Keys 페이지 - 새로운 비밀 키를 생성 
    * 발급받은 API 키는 비밀로 관리해야 하며, 소스 코드나 공개 저장소에 노출되지 않도록 주의
        * API 키를 코드에 하드코딩하지 않고, 환경 변수나 별도의 설정 파일(.env)에 저장

- .env 파일에 키 저장 : 프로젝트 디렉터리에 .env 파일을 만들고 API 키를 저장

    - OPENAI_API_KEY=발급받은-API키-값

- python-dotenv 사용: 파이썬 코드에서 python-dotenv를 이용해 .env 파일의 환경 변수를 불러오기



In [ ]:
import openai
openai.__version__

In [ ]:
from dotenv import load_dotenv
load_dotenv()


# 코랩에서 실행 시 : 보안 비밀 키 - 새 보안 비밀 추가 - 이름/값 설정
# from google.colab import userdata
# userdata.get('secretName')
# from openai import OpenAI
# client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

In [ ]:
import os
os.getenv('OPENAI_API_KEY')[:3]

In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-4o-mini",
    input="What is gpt?"
)

print(response.output_text)

In [ ]:
response.output[0].content[0].text

In [ ]:
response.usage

In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-4o-mini",
    input="gpt가 뭐야?"
)

print(response.output_text)

In [ ]:
response.usage

In [ ]:
# 추론 모델
response = client.responses.create(
    model="gpt-5-nano",
    input="gpt가 뭐야?"
)

print(response.output_text)

In [ ]:
response.usage

### 추론 / 비추론 모델
- 추론 모델 : o 시리즈 (o1, o3, o3-mini), GPT-5 시리즈 (gpt-5, gpt-5-mini, gpt-5-nano)
- 비추론 모델 : GPT-4 시리즈 이하 (gpt-4o, gpt-4.1), Claude 3.5 Sonnet, Gemini 1.5 Flash

# 2. 기본적인 Chat Completions API 사용법

## 2.1 간단한 텍스트 생성 요청
- Chat Completions 엔드포인트 : 메시지 목록을 입력으로 받아 다음에 이어질 메시지를 생성
- 각 메시지는 role과 content 필드로 구성 : 일반적으로 **user (사용자 메시지), assistant (모델의 응답 메시지), system (시스템 지시 메시지)** 세 가지 역할을 사용

In [ ]:
# 비추론 모델 사용자 메세지 구성
messages = [
    {'role':'user', 'content':'안녕하세요. 오늘 날씨가 어떤가요?'}
]

response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = messages,
    temperature = 0.7, # 창의성 조절 : 0.0 ~ 2.0 사이의 값으로 설정
    frequency_penalty = 0.5 # 단어 빈도 감점 : -2.0 ~ 2.0 사이의 값으로 설정
)

In [ ]:
response.choices[0].message.content

In [ ]:
# 추론 모델 사용자 메세지 구성
messages = [
    {'role':'user', 'content':'안녕하세요. 오늘 날씨가 어떤가요?'}
]

response = client.chat.completions.create(
    model = 'gpt-5-nano',
    messages = messages,
    reasoning_effort = 'minimal' # 추론 깊이 조절 : minimal, low, medium, high
)

In [ ]:
response.choices[0].message.content

In [ ]:
# 비추론 모델 few-shot : input token 사용량 증가 유의
response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = [
        {'role':'system', 'content':'친절하게 답변해주는 비서'}, # 역할 부여 (페르소나 설정)
        {'role':'user', 'content':'프랑스의 수도는?'}, # few shot
        {'role':'assistant', 'content':'파리(수도명만 대답)'},
        {'role':'user', 'content':'이탈리아의 수도는 어디야?'}, # few shot
        {'role':'assistant', 'content':'로마(수도명만 대답)'},
        {'role':'user','content':'한국의 수도는?'}
    ]
)

response

## 2.2 client
- 이전 대화나 답변을 기억하는 내장 메모리(상태)를 가지고 있지 않음 : API는 완전한 Stateless(무상태) 방식으로 동작
- AI가 이전 대화를 기억하게 하려면, 개발자가 직접 이전 response의 답변을 messages 리스트에 포함시켜 매번 재전달 필요

In [ ]:
# 비추론 모델 역할 설정 1
response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = [
        {'role':'system', 'content':'너는 컴퓨터 프로그램 전문가야'}, # 역할 부여 (페르소나 설정)
        {'role':'user','content':'Spring이 뭐야?'}
    ], 
    temperature = 0.7,
    frequency_penalty = 0.5
)

print(response.choices[0].message.content)

In [ ]:
# 비추론 모델 역할 설정 2
response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = [
        {'role':'system', 'content':'너는 시를 좋아하는 문학 전문가야'}, # 역할 부여 (페르소나 설정)
        {'role':'user','content':'Spring이 뭐야?'}
    ], 
    temperature = 0.7,
    frequency_penalty = 0.5
)

print(response.choices[0].message.content)

In [ ]:
# 비추론 모델 역할 설정 3
response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = [
        {'role':'system', 'content':'너는 친절하지만 짧게 대답해주는 비서야'}, # 역할 부여 (페르소나 설정)
        {'role':'user','content':'2020년 월드시리즈 누가 우승했어?'}
    ], 
    temperature = 1,
    frequency_penalty = 0.5,
    # max_tokens = 100
)

print(response.choices[0].message.content)

In [ ]:
# 비추론 모델 역할 설정 3-1 ; 이전 답변을 포함하여 답변하도록 지시
response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = [
        {'role':'system', 'content':'너는 친절하지만 짧게 대답해주는 비서야'}, # 역할 부여 (페르소나 설정)
        {'role':'user','content':'2020년 월드시리즈 누가 우승했어?'},
        {'role':'assistant','content':'2020년 월드시리즈는 로스앤젤레스 다저스가 우승했어요.'},
        {'role':'user','content':'점수와 상대 팀은?'}
    ], 
    temperature = 1,
    frequency_penalty = 0.5
)

print(response.choices[0].message.content)

In [ ]:
# 비추론 모델 역할 설정 4 ; 이전 답변을 포함하여 답변하도록 지시
response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = [
        {'role':'system', 'content':'너는 친절하지만 짧게 대답해주는 비서야'}, # 역할 부여 (페르소나 설정)
        {'role':'user','content':'2002년 월드컵에서 가장 화제가 되었던 나라는 어디야?'},
        {'role':'assistant','content':'예상을 뚫고 4강에 진출한 한국입니다.'},
        {'role':'user','content':'화제가 된 이유를 100자 이내로 설명해줘'}
    ]
)

print(response.choices[0].message.content)

In [ ]:
# JSON 형태의 output 받기
response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = [
        {'role':'system', 'content':'너는 친절하지만 짧게 대답해주는 비서야, JSON으로 출력해줘'}, #  response_format={'type':'json_object'}일 경우, 프롬프트에 JSON이 포함되어야함
        {'role':'user','content':'2020년 월드시리즈 누가 우승했어?'}
    ], 
    response_format={'type':'json_object'}, # JSON 형태로 응답 (기본값 : text)
    temperature = 1,
    frequency_penalty = 0.5
)

json_string = response.choices[0].message.content
print(json_string) # json 형태의 문자열

In [ ]:
# json 형태의 문자열을 json 데이터로 변환
import json
data = json.loads(json_string)
print(data)

In [ ]:
# 웹에서 활용 시 함수화 필요
from dotenv import load_dotenv
from openai import OpenAI

def askGPT(prompt):
    'gpt-4.1-nano의 prompt 요청 결과를 반환'
    load_dotenv()
    client = OpenAI()
    response = client.chat.completions.create(
        model = 'gpt-4.1-nano',
        messages = [
            {'role':'system','content':'당신은 텍스트를 요약하는 어시스턴트 입니다.'},
            {'role':'user','content':f'your task is to summarize the text sentences in korean language. Summarize in 2 lines. Use the format of a check(✅). text : {prompt}'}
        ] 
    )
    return response.choices[0].message.content

In [ ]:
article = input('요약할 글을 입력하세요.')
print(askGPT(article))

# 3. 스트리밍 응답 (Streaming)
- 스트리밍 활용 시 사용자에게 실시간으로 응답을 표시하거나, 매우 긴 응답을 토큰 단위로 처리 가능

## 3.1 스트리밍이 필요한 경우
- 실시간 피드백 : 사용자 경험을 개선하기 위해 답변 생성을 기다리는 동안 실시간으로 텍스트를 보여줄 때
- 긴 응답 처리 : 응답이 길어서 한꺼번에 받으면 메모리 사용이 많을 때, 토큰이 도착하는 대로 처리 가능
- 중간 작업 가능 : 응답을 받는 도중에도 다른 이벤트를 처리하거나 UI 업데이트를 할 수 있음

## 3.2 스트리밍 사용 방법
- OpenAI 파이썬 라이브러리에서 스트리밍을 사용 : 요청 시 stream=True 옵션
- 응답 객체 대신 **이터레이터(iterator)** 를 반환하며, 이 이터레이터를 순회(for 문 등)하면서 부분 응답(chunk)을 받을 수 있음

In [ ]:
# 스트리밍 응답
import time
messages = [
    {'role':'system','content':'대한민국을 사랑하는 도우미 입니다. 대한민국의 도시 이름만 한글자씩 출력하세요'},
    {'role':'user','content':'도시명 5개 알려줘'}
]

for chunk in client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = messages,
    stream = True
):
    # 스트리밍으로 들어온 chunk에서 content만 추출
    chunk_msg = chunk.choices[0].delta.content
    if chunk_msg:
        print(chunk_msg, end='/')
        time.sleep(.5)

```위 코드를 실행하면 response_stream은 응답 스트림 객체가 되고, for 루프에서 순차적으로 응답 조각을 받아옵니다. 각 chunk는 choices[0].delta에 현재 추가 생성된 텍스트 조각을 담고 있습니다 (완전한 메시지가 아니라 추가된 부분만을 담음). 이를 이어붙여 화면에 출력하면 모델이 답변을 조금씩 생성해가는 과정을 실시간으로 볼 수 있습니다. 예를 들어, 모델이 "안녕하세요, 만나서 반갑습니다."라는 문장을 생성한다면, 스트리밍 출력은 사람이 타이핑하듯 안, 안녕, 안녕하세요, ... 차례로 출력될 것입니다. 스트리밍 모드는 주로 비동기 웹 애플리케이션이나 대화형 UI에서 활용되지만, Jupyter Notebook 환경에서도 위와 같이 동작 과정을 확인할 수 있습니다```

# 4. 시스템 메시지 활용
- 시스템 메시지(system role message) : 모델에게 전체 대화의 맥락이나 규칙을 알려주는 역할
    - AI의 말투, 행동 방식, 응답 형식 등을 조정
    - 대화의 첫 번째 메시지로 넣는 경우가 많으며, 사용자에게는 보이지 않지만 모델에게는 강한 지침으로 작용

## 시스템 메시지의 역할
- 행동 지침 : 모델이 따라야 할 규칙이나 목표를 제시 (예: "반말로 대답하지 마세요", "모든 응답에 이모티콘 하나를 포함하세요").
- 역할 부여 : 모델에게 특정 인격이나 역할을 부여 (예: "너는 역사 전문가야", "너는 사용자를 돕는 비서야").
- 컨텍스트 설정 : 대화 주제나 맥락을 사전에 설정 (예: "이 대화는 의료 상담입니다", "사용자는 프로그래밍 도움을 요청할 것입니다").

In [ ]:
messages = [
    {'role':'system','content':'You are a coding assistant that talks like a pirate'},
    {'role':'user','content':'Python에서 객체가 특정 클래스의 인스턴스인지 확인하려면 어떻게 하는지 한국어로 설명해줘.'}
]

response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = messages,
    temperature = 1,
    frequency_penalty = 0.5
)

print(response.choices[0].message.content)

In [ ]:
messages = [
    {'role':'system','content':'You are a coding assistant that talks like a grandma'},
    {'role':'user','content':'Python에서 객체가 특정 클래스의 인스턴스인지 확인하려면 어떻게 하는지 한국어로 설명해줘.'}
]

response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = messages,
    temperature = 1,
    frequency_penalty = 0.5
)

print(response.choices[0].message.content)

```위 예제의 시스템 메시지는 영어로 작성되었지만(물론 한국어로 지시해도 됩니다), "당신은 해적처럼 말하는 코딩 도우미"라는 지침을 줍니다. 그 다음 사용자 질문은 일반적으로 "Python에서 객체가 특정 클래스의 인스턴스인지 어떻게 확인하나요?"라는 내용입니다. 시스템 메시지 덕분에, 모델의 답변은 아마도 해적 말투로 나올 것입니다.

이처럼 동일한 질문이라도 시스템 메시지를 통해 모델의 답변 스타일이나 관점을 크게 바꿀 수 있습니다. 필요에 따라 시스템 메시지를 활용하여 프로젝트의 톤앤매너에 맞는 응답을 얻도록 조정하세요.
```

> 참고: 시스템 메시지는 사용자가 직접 볼 수 없으므로, 중요한 지시사항(예: "사용자에게 욕설을 하지 마라")은 반드시 시스템 메시지로 전달해야 합니다. 모델은 사용자 메시지의 내용보다 시스템 메시지의 지시에 우선순위를 두도록 설계되어 있습니다.


# 5. 고급 활용법
- 토큰 사용을 최적화하여 비용을 절감하는 방법
- API 호출 시 발생할 수 있는 오류를 처리하는 방법

## 5.1 토큰 최적화 및 비용 절감
- OpenAI API 비용은 사용한 토큰(token) 수에 비례하여 청구됨
- 동일한 작업을 하더라도 토큰을 적게 사용하면 비용이 줄어들고, 응답 속도도 향상됨

- 토큰 최적화를 위한 팁
    - 짧고 명확한 프롬프트 : 시스템 메시지와 사용자 메시지를 간결하게 작성
    - 대화 내역 관리 : 이전 대화 기록을 얼마나 포함시킬지 결정 (중요한 맥락만 남기고 요약하거나 일부 생략)
    - 모델 선택 : 높은 성능이 필요하지 않은 작업에는 같은 더 작은 모델을 사용해 비용을 절감
    - max_tokens 파라미터 활용 : 응답의 최대 길이를 설정 및 제어
    
- 스트리밍과 부분 처리 : 매우 긴 응답의 경우 중간 중간 출력 결과를 확인하며 필요에 따라 조기에 중단하는 등의 대응 가능

- OpenAI는 Batch API 등을 통해 다수의 요청을 한 번에 보내 비용을 절약하는 방법 제공

- 토큰 최적화의 효과를 확인 : 응답 객체의 usage 정보 출력
    - response.usage : prompt_tokens(입력 토큰 수), completion_tokens(출력 토큰 수), total_tokens(합계)

In [ ]:
response.usage

In [ ]:
response.usage.completion_tokens, response.usage.prompt_tokens

```이런 정보를 토대로 모델이 과도하게 긴 답변을 내놓지는 않았는지 모니터링하고, 프롬프트를 조정하는 피드백 loop을 거치면 점점 효율적으로 API를 활용할 수 있습니다.```

## 5.2 에러 핸들링 및 예외 처리 요약

* 네트워크 오류 및 타임아웃
    * 인터넷 연결 문제 및 서버 응답 지연으로 인한 요청 실패
    * 요청 재시도 및 지수적 지연 전략(exponential backoff) 적용


* 레이트 리미트(Rate Limit) 초과
    * 허용 요청량 초과로 인한 `RateLimitError` 발생
    * 일정 시간 대기 후 재시도 및 요청 빈도 조절


* 유효하지 않은 요청
    * 모델명 오타, 매개변수 형식 오류, DALL-E 이미지 처리 오류 등으로 인한 `InvalidRequestError` 발생
    * API 호출 전 사전 코드 검증


* API 키 오류
    * 잘못된 API 키 및 권한 문제로 인한 인증 오류(`AuthenticationError`) 발생
    * API 키 유효성 및 권한 사전 확인


* Python 라이브러리 예외 구조
    * 최상위 부모 예외 클래스 : `openai.error.OpenAIError`
    * 하위 예외 클래스 : `openai.error` 모듈 내 세부 예외 클래스로 관리

In [ ]:
import openai
try:
    res = client.chat.completions.create(
        model = 'gpt-4.1-nano',
        messages = [{'role':'user','content':'에러 발생시키기'}],
        timeout = 0.01 # 답변 제한시간 (초과 시 에러)
    )
    
# 모든 openai 에러는 OpenAIError에서부터 상속 받음
except openai.OpenAIError as e:
    print('요청 시간을 초과하였습니다.')
    print(e)
    print(type(e))

```
위 코드에서 timeout=5는 응답이 5초 안에 없으면 OpenAIError를 발생시키도록 한 것으로, 강제로 타임아웃 상황을 연출하기 위한 예시입니다. RateLimitError는 별도로 캐치하여 사용자에게 요청 제한 메세지를 보여주고, 그 외 모든 OpenAI 오류는 일반적으로 메시지(e)를 출력하도록 했습니다. 실제 애플리케이션에서는 오류 종류에 따라 로깅을 남기고, 필요하면 재시도 로직을 넣는 등 더 정교한 대응을 구현할 수 있습니다.

마지막으로, 예상하지 못한 예외 상황(예: JSON 디코딩 오류나 타입 오류 등)이 발생할 수 있으므로, API 호출 코드 주위에는 일반 예외 처리도 넣어서 프로그램이 갑자기 중단되지 않도록 만드는 것이 좋습니다.
```

# 실전 프로젝트 예제

## 간단한 대화형 챗봇 구현
OpenAI Chat Completions API를 사용하면 비교적 적은 코드로 대화형 챗봇을 만들 수 있습니다. <br>
여기서는 콘솔에서 사용자의 입력을 받아 모델의 응답을 출력하는 간단한 챗봇을 구현해봅니다. <br>
이 챗봇은 이전 대화 맥락을 기억하여 연속적인 대화를 주고받을 수 있습니다.


In [ ]:
chat_history = [
    {'role':'system','content':'당신은 유능한 AI 상담원 입니다.'},
]

print('챗봇 시작 (종료 : exit, 나가기)')
input_tokens = 0
out_tokens = 0

while True:
    user_input = input('사용자 : ').strip()
    
    if user_input.lower() in ['exit', '나가기']:
        print('입력 토큰 :', input_tokens)
        print('출력 토큰 :', out_tokens)
        print('총 비용 :', ((input_tokens + out_tokens*4)/1000000)*0.1, '$')
        print('챗봇 종료')
        break
    
    if user_input.strip() == '':
        continue
        
    chat_history.append(
        {'role':'user','content':user_input}
    )
    
    try:
        # openai API 호출
        response = client.chat.completions.create(
            model = 'gpt-4.1-nano',
            messages = chat_history
        )
    except openai.OpenAIError as e:
        print('오류가 발생하였습니다.')
        break
        
    
    assistant_reply = response.choices[0].message.content.strip()
    print('AI 답변 :', assistant_reply)
    
    # 토큰 최적화 : chat_history의 내용을 요약하여 추가하는 작업
    
    chat_history.append(
        {'role':'assistant','content':assistant_reply}
    )
    
    input_tokens += response.usage.completion_tokens
    out_tokens += response.usage.prompt_tokens